
# Card &middot; Multitask learning with Chemprop

| | |
|---|---|
| **time** | ~60 minutes |
| **GPU** | **yes** &mdash; Runtime &rarr; Change runtime type &rarr; T4 |
| **typical gain** | large, especially on the data-poor endpoints |
| **needs** | nothing; runs standalone |

Multitask graph neural networks won this challenge outright. Chemprop appeared
in most of the top 20, and **every single one of the top five** trained
endpoints together rather than separately.

The idea: instead of nine separate models, train one model with a shared
molecular encoder and nine output heads. The encoder has to learn a
representation of a molecule that is useful for all nine tasks at once, and
the endpoints with lots of data end up paying for a better representation that
the data-poor endpoints get to use for free.

### Check your runtime first

In [ ]:
import torch
print("GPU available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    print("\n>>> Runtime -> Change runtime type -> T4 GPU, then re-run. <<<")

In [ ]:
# Run me first.
%pip -q install rdkit pandas numpy scipy scikit-learn huggingface_hub fsspec chemprop

# Get common.py. If you uploaded it yourself (folder icon in the left sidebar),
# this leaves your copy alone -- it only downloads when the file is missing.
!test -s common.py || wget -q -O common.py https://raw.githubusercontent.com/CHANGE-ME/admet-hackathon/main/common.py

import os, sys
assert os.path.exists("common.py") and os.path.getsize("common.py") > 1000, (
    "common.py is missing or truncated. Upload it using the folder icon in the "
    "left sidebar, then re-run this cell.")

sys.modules.pop("common", None)   # force a fresh read if you just re-uploaded it
import common
common.setup(pair="CHANGE-ME")

In [ ]:
import numpy as np, pandas as pd, os, subprocess, sys
train = common.load_train()
test  = common.load_test()
fold, split_meta = common.load_split(train)
print("split in use:", split_meta.get("method"))
WORK = common.workdir()

---
## 1. Sparsity is handled for you (and this is the key detail)

Your target matrix is mostly `NaN`. Chemprop does not drop those rows &mdash; it
**masks** them. For each molecule, the loss is computed only over the endpoints
that were actually measured, and no gradient flows from the missing ones.

That masking is the entire reason sparse multitask learning works. A molecule
with only an HLM measurement still contributes: it trains the shared encoder,
which improves the representation used by all nine heads. Under nine separate
models, that molecule would have been invisible to eight of them.

In [ ]:
def write_chemprop_csv(df, path, endpoints):
    cols = [common.SMILES_COL] + list(endpoints)
    out = df[[c for c in cols if c in df.columns]].copy()
    for e in endpoints:
        if e not in out.columns:
            out[e] = np.nan
    out.to_csv(path, index=False)
    return path


def run_chemprop(endpoints, tag, epochs=30, extra_args=()):
    """Train a Chemprop model on `endpoints`, return validation metrics."""
    d = os.path.join(WORK, "chemprop", tag)
    os.makedirs(d, exist_ok=True)
    tr_path = write_chemprop_csv(train[(fold == "train").to_numpy()],
                                 os.path.join(d, "train.csv"), endpoints)
    va_path = write_chemprop_csv(train[(fold == "val").to_numpy()],
                                 os.path.join(d, "val.csv"), endpoints)

    cmd = ["chemprop", "train",
           "--data-path", tr_path,
           "--separate-val-path", va_path,
           "--separate-test-path", va_path,
           "--task-type", "regression",
           "--smiles-columns", common.SMILES_COL,
           "--target-columns", *endpoints,
           "--output-dir", d,
           "--epochs", str(epochs),
           "--num-workers", "0",
           *extra_args]
    print(" ".join(cmd[:6]), "...")
    subprocess.run(cmd, check=True)
    return d


def chemprop_predict(model_dir, df, endpoints, tag):
    d = os.path.join(model_dir, "pred_" + tag)
    os.makedirs(d, exist_ok=True)
    inp = write_chemprop_csv(df, os.path.join(d, "input.csv"), endpoints)
    outp = os.path.join(d, "preds.csv")
    ckpt = os.path.join(model_dir, "model_0", "best.pt")
    subprocess.run(["chemprop", "predict",
                    "--test-path", inp,
                    "--model-path", ckpt,
                    "--preds-path", outp,
                    "--smiles-columns", common.SMILES_COL], check=True)
    p = pd.read_csv(outp)
    out = pd.DataFrame({common.ID_COL: df[common.ID_COL].to_numpy()})
    for e in endpoints:
        col = e if e in p.columns else [c for c in p.columns if e in c][0]
        out[e] = p[col].to_numpy()
    return out

> **Instructor note:** Chemprop's CLI flags shift between minor
> versions. Run one 2-epoch job the week before the event and fix the
> arguments above if anything has moved. The `best.pt` path in particular has
> changed in the past.

---
## 2. Does sharing actually help?

The cleanest experiment in this notebook. Take one **data-poor** endpoint and
train it two ways: alone, and grouped with LogD (which has the most data of
any endpoint).

`Log_Mouse_BPB` is a good choice &mdash; few measurements, and physically it is
driven by lipophilicity, which is exactly what LogD measures.

### &#9654;&#65039; Predict first

**Will training Log_Mouse_BPB together with LogD help it, hurt it, or do nothing? And what happens to LogD itself?**

Write your answer here before running the next cell &mdash; one line is enough:

> `your prediction:`

In [ ]:
POOR = "Log_Mouse_BPB"
va_df = train[(fold == "val").to_numpy()].reset_index(drop=True)

d_solo = run_chemprop([POOR], "solo", epochs=30)
p_solo = chemprop_predict(d_solo, va_df, [POOR], "val")
ev_solo = common.evaluate(va_df, p_solo, [POOR])

d_pair = run_chemprop(["LogD", POOR], "with_logd", epochs=30)
p_pair = chemprop_predict(d_pair, va_df, ["LogD", POOR], "val")
ev_pair = common.evaluate(va_df, p_pair, ["LogD", POOR])

print("\nalone      :", ev_solo.loc[POOR, "RAE"].round(3))
print("with LogD  :", ev_pair.loc[POOR, "RAE"].round(3))
print("LogD itself:", ev_pair.loc["LogD", "RAE"].round(3))

In the real challenge, several top finishers trained the protein
binding endpoints together with LogD for exactly this reason &mdash; the binding
models get to use feature representations learned from the much larger LogD
dataset.

Check the second number too. Did helping the small endpoint *cost* the large
one anything? Sharing is not free, and when tasks conflict, the shared encoder
has to compromise.

---
## 3. Which endpoints should be grouped?

You cannot test every option. The number of ways to partition nine endpoints
into groups is 21,147, and even the narrower question "which subset joins
LogD" has 256 answers. This is a real constraint, not a notebook limitation:
project teams face the same wall.

So you need a hypothesis. Three sources:

1. **The correlation heatmap** from `01_eda`. Remember its caveats &mdash; sparse
   overlap makes some of those correlations meaningless, and label correlation
   is not the same thing as task affinity.
2. **Chemistry.** Lipophilicity drives permeability and protein binding. The
   two clearance assays are the same experiment in two species. The two Caco-2
   readouts come off the same plate.
3. **What worked before.** Task-affinity grouping has a literature, and
   the top finishers used it.

`common.ENDPOINT_FAMILIES` gives you a chemistry-based starting point. You have
compute for maybe three or four groupings. Choose deliberately.

In [ ]:
common.ENDPOINT_FAMILIES

In [ ]:
GROUPINGS = {
    "all_together": [common.ENDPOINTS],
    "by_family":    list(common.ENDPOINT_FAMILIES.values()),
    "your_idea":    [["LogD", "Log_Mouse_PPB", "Log_Mouse_BPB", "Log_Mouse_MPB"],
                     ["LogS"],
                     ["Log_HLM_CLint", "Log_MLM_CLint"],
                     ["Log_Caco_Papp_AB", "Log_Caco_ER"]],
}

def evaluate_grouping(name, groups, epochs=30):
    parts = []
    for i, g in enumerate(groups):
        d = run_chemprop(g, f"{name}_{i}", epochs=epochs)
        parts.append(chemprop_predict(d, va_df, g, "val").set_index(common.ID_COL))
    merged = pd.concat(parts, axis=1).reset_index()
    ev = common.evaluate(va_df, merged)
    print(f"\n{name}: MA-RAE = {ev['RAE'].mean():.3f}")
    return ev, merged

ev_all, pred_all = evaluate_grouping("all_together", GROUPINGS["all_together"])

In [ ]:
# Try one or two more. Each grouping costs one training run per group,
# so "by_family" is four runs. Budget accordingly.
ev_fam, pred_fam = evaluate_grouping("by_family", GROUPINGS["by_family"])

pd.DataFrame({"all_together": ev_all["RAE"], "by_family": ev_fam["RAE"]}).round(3)

---
## 4. CheMeleon: start from a model that already knows chemistry

`CheMeleon` is a foundation model: a Chemprop encoder pre-trained on a large
corpus so that it arrives already knowing how to represent molecules. You
fine-tune from there instead of from random weights.

At least eight of the top twenty finishers used it, making it the single most
popular pre-training choice in the challenge.

In [ ]:
# Fine-tune from the CheMeleon foundation weights.
# INSTRUCTOR: verify this flag against the chemprop version you pin --
# see https://github.com/JacksonBurns/chemeleon
d_chem = run_chemprop(common.ENDPOINTS, "chemeleon", epochs=30,
                      extra_args=("--from-foundation", "CheMeleon"))
p_chem = chemprop_predict(d_chem, va_df, common.ENDPOINTS, "val")
ev_chem = common.evaluate(va_df, p_chem)

pd.DataFrame({"from scratch": ev_all["RAE"],
              "from CheMeleon": ev_chem["RAE"]}).round(3)

---
## A note on hard vs. soft parameter sharing

What Chemprop does is **hard sharing**: one encoder, shared by every task, with
separate output heads. Every task sees the same molecular representation.

**Soft sharing** gives each task its own encoder and adds a penalty encouraging
them to stay similar &mdash; more flexible when tasks partly conflict, but it is
not built into Chemprop and there is no evidence from this challenge that it
was needed. Every top finisher used hard sharing.

Mentioned so you know the word. Do not build it today.

---
## Save your work

Give it a name you will recognise at 4pm. `card_ensembles` can combine this
with anything else you have made today.

In [ ]:
BEST = "all_together"     # <-- your pick

groups = GROUPINGS[BEST]
parts = []
for i, g in enumerate(groups):
    d = run_chemprop(g, f"final_{i}", epochs=40)
    parts.append(chemprop_predict(d, test, g, "test").set_index(common.ID_COL))
pred = pd.concat(parts, axis=1).reset_index()

common.save_predictions(pred, f"chemprop-{BEST}",
                        note=f"Chemprop v2 multitask, grouping={BEST}")